In [ ]:
!pip install -U bitsandbytes transformers accelerate datasets peft
!pip install --upgrade torchao>=0.16.0

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
import torch
import os
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

os.makedirs("data", exist_ok=True)

print("Загрузка данных...")

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")

print(f"Поля датасета: {dataset.column_names}")
print(f"Всего примеров: {len(dataset)}")

# 500 примеров для обучения
dataset = dataset.select(range(500))

def format_instruction(example):
    instruction_text = example.get('instruction', '')
    response_text = example.get('response', '')
    formatted = f"[INST] {instruction_text} [/INST] {response_text}</s>"
    return {"text": formatted}

print("Форматирование данных...")
formatted_dataset = dataset.map(format_instruction)

print(f"Подготовлено {len(formatted_dataset)} примеров")

# Настройка модели
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Загрузка модели
if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu",
        low_cpu_mem_usage=True
    )

# LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Query, Value, Key, Output
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print(f"trainable params: {model.num_parameters(only_trainable=True):,} / {model.num_parameters():,}")

# Токенизация
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized



print("Токенизация...")
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Настройки обучения
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4 if device == "cuda" else 1,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    fp16=device == "cuda",
    dataloader_num_workers=2,
)

# Создание Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)



trainer.train()
trainer.save_model("./results/bitext_support_bot")
tokenizer.save_pretrained("./results/bitext_support_bot")
print("Модель сохранена в ./results/bitext_support_bot")
print("\nТестируем модель:")
model.eval()

test_prompts = [
    "[INST] How can I return a product? [/INST]",
    "[INST] Where is my order? [/INST]",
    "[INST] I have a problem with payment [/INST]",
]

for test_prompt in test_prompts:
    inputs = tokenizer(test_prompt, return_tensors="pt")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nЗапрос: {test_prompt}")
    print(f"Ответ: {response}")
    print("-" * 50)

print("\nОбучение и тестирование завершены!")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')


full_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
test_dataset = full_dataset.select(range(500, 505))  # 5 примеров

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()

finetuned_model = AutoModelForCausalLM.from_pretrained(
    "./results/bitext_support_bot",
    torch_dtype=torch.float16,
    device_map="auto"
)
finetuned_model.eval()

def ask(model, question):
    prompt = f"[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.replace(prompt, "").strip()

for i, example in enumerate(test_dataset):
    question = example['instruction']

    print(f"ВОПРОС {i+1}: {question}")

    print(f"\nЭТАЛОН: {example['response'][:150]}...")
    print(f"\nДО ОБУЧЕНИЯ: {ask(base_model, question)[:200]}")
    print(f"\nПОСЛЕ ОБУЧЕНИЯ: {ask(finetuned_model, question)[:200]}")
    print()


In [ ]:
!pip install langchain langgraph

In [ ]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from langchain.tools import tool
import warnings
warnings.filterwarnings('ignore')

print("Загрузка модели...")

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Базовая модель для сравнения
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()

# Дообученная модель
finetuned_model = AutoModelForCausalLM.from_pretrained(
    "./results/bitext_support_bot",
    torch_dtype=torch.float16,
    device_map="auto"
)
finetuned_model.eval()

print("Модели загружены")


DATABASE = {
    "orders": {
        "ORD-001": {
            "order_id": "ORD-001",
            "customer_name": "John Smith",
            "customer_email": "john@example.com",
            "status": "shipped",
            "total": 125.50,
            "items": [
                {"product": "Laptop", "quantity": 1, "price": 99.99},
                {"product": "Mouse", "quantity": 2, "price": 12.75}
            ],
            "created_at": "2026-06-15",
            "tracking": "TRK-001"
        },
        "ORD-002": {
            "order_id": "ORD-002",
            "customer_name": "Maria Johnson",
            "customer_email": "maria@example.com",
            "status": "processing",
            "total": 67.50,
            "items": [
                {"product": "Headphones", "quantity": 1, "price": 45.00},
                {"product": "Case", "quantity": 3, "price": 7.50}
            ],
            "created_at": "2026-06-16"
        },
        "ORD-003": {
            "order_id": "ORD-003",
            "customer_name": "Alex Brown",
            "customer_email": "alex@example.com",
            "status": "delivered",
            "total": 34.00,
            "items": [
                {"product": "Book", "quantity": 2, "price": 12.00},
                {"product": "Stationery", "quantity": 4, "price": 2.50}
            ],
            "created_at": "2026-06-10"
        }
    }
}


@tool
def get_order_status(order_id: str) -> str:
    """
    Get the status of an order by its ID.
    Args:
        order_id: Order number (e.g., ORD-001)
    Returns: JSON with order status information
    """
    order = DATABASE["orders"].get(order_id)
    if not order:
        return json.dumps({"error": f"Order {order_id} not found"})

    return json.dumps({
        "order_id": order["order_id"],
        "status": order["status"],
        "total": order["total"],
        "items_count": len(order["items"]),
        "customer_name": order["customer_name"],
        "created_at": order["created_at"]
    })

@tool
def get_order_details(order_id: str) -> str:
    """
    Get order's details by its ID.
    Args:
        customer_id: Order number (e.g., ORD-001)
    Returns: JSON with list of order's details
    """
    order = DATABASE["orders"].get(order_id)
    if not order:
        return json.dumps({"error": f"Order {order_id} not found"})

    return json.dumps(order)

@tool
def cancel_order(order_id: str, reason: str = "not specified") -> str:
    """
    Cancel an order by its ID.
    Args:
        order_id: Order number
    Returns: JSON with cancellation status
    """
    order = DATABASE["orders"].get(order_id)
    if not order:
        return json.dumps({"error": f"Order {order_id} not found"})

    if order["status"] in ["shipped", "delivered"]:
        return json.dumps({
            "error": f"Order {order_id} is already {order['status']} and cannot be cancelled"
        })

    order["status"] = "cancelled"
    return json.dumps({
        "success": True,
        "message": f"Order {order_id} has been successfully cancelled",
        "refund_amount": order["total"]
    })

TOOLS = [get_order_status, get_order_details, cancel_order]

for tool in TOOLS:
    print(f"  • {tool.name}: {tool.description}")


def ask_base(model, question):
    prompt = f"[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.replace(prompt, "").strip()

def ask_with_tools(model, question, use_tools=True):
    if use_tools:
        order_match = re.search(r'ORD-\d{3}', question)

        if any(word in question.lower() for word in ['status', 'where', 'track']):
            if order_match:
                order_id = order_match.group()
                result = json.loads(get_order_status.invoke({"order_id": order_id}))
                if "error" not in result:
                    return (f"Order {result['order_id']}\n"
                           f"Status: {result['status'].upper()}\n"
                           f"Total: ${result['total']:.2f}\n"
                           f"Items: {result['items_count']}\n"
                           f"Customer: {result['customer_name']}\n"
                           f"Created: {result['created_at']}")

        if any(word in question.lower() for word in ['details', 'full', 'items', 'show']):
            if order_match:
                order_id = order_match.group()
                result = json.loads(get_order_details.invoke({"order_id": order_id}))
                if "error" not in result:
                    items_str = "\n".join([
                        f"{item['product']} x{item['quantity']}: ${item['price']:.2f}"
                        for item in result['items']
                    ])
                    return (f"Order {result['order_id']} Details\n"
                           f"Customer: {result['customer_name']}\n"
                           f"Email: {result['customer_email']}\n"
                           f"Status: {result['status'].upper()}\n"
                           f"Total: ${result['total']:.2f}\n"
                           f"Items:\n{items_str}\n"
                           f"Created: {result['created_at']}")

        if 'cancel' in question.lower():
            if 'confirm' in question.lower() and order_match:
                order_id = order_match.group()
                result = json.loads(cancel_order.invoke({
                    "order_id": order_id,
                    "reason": "customer requested"
                }))
                if "error" not in result:
                    return f"{result['message']}\nRefund: ${result['refund_amount']:.2f}"
            elif order_match:
                order_id = order_match.group()
                return f"Confirm cancellation of {order_id}? Reply with: 'confirm cancel {order_id}'"

    return ask_base(model, question)

def ask(model, question):
    return ask_base(model, question)



print("\n" + "="*60)
print("Тестируем каждый тул отдельно")
print("="*60)

test_cases = [
    ("get_order_status", {"order_id": "ORD-001"}),
    ("get_order_details", {"order_id": "ORD-001"}),
    ("cancel_order", {"order_id": "ORD-002", "reason": "changed mind"}),
]

for tool_name, args in test_cases:
    print(f"\n{tool_name}({args})")
    if tool_name == "get_order_status":
        result = get_order_status.invoke(args)
    elif tool_name == "get_order_details":
        result = get_order_details.invoke(args)
    elif tool_name == "cancel_order":
        result = cancel_order.invoke(args)
    print(f"{json.loads(result)}")


# Тестовые вопросы с заказами
test_questions = [
    "What is the status of order ORD-001?",
    "Show me details of order ORD-001",
    "I want to cancel order ORD-002",
    "What is your return policy?",
    "How can I track my order?"
]
print("\n" + "="*60)
print("\nТестовые вопросы с заказами:\n")
print("="*60)

for i, question in enumerate(test_questions):
    print(f"ВОПРОС {i+1}: {question}")

    print("\nБез тулов(базовая модель):")
    response_base = ask_base(base_model, question)
    print(f"{response_base[:150]}...")

    print("\nС тулами(дообученная модель):")
    response_tools = ask_with_tools(finetuned_model, question, use_tools=True)
    print(f"{response_tools[:150]}...")

    print("\n" + "-"*60)

